# 03. Parking & Aerial Imagery EDA
**Task 1.2.3 - Parking Proxies & Sensor Evaluation**

### Objective:
- Conduct an EDA on historical parking sensor records to evaluate baseline and post-intervention parking utilization in the City of Melbourne.
- Prototype an aerial imagery object detection or sampling methodology to count parked cars in buffer zones for LGAs lacking sensor infrastructure.


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import yaml
from pathlib import Path

# Setup plotting styles
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style="whitegrid")

# Load project configuration
config_path = Path("../config.yaml")
if config_path.exists():
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    print("Project configuration loaded successfully.")
    print("Data Paths:", config.get('paths', {}))
else:
    print("Warning: config.yaml not found at", config_path.resolve())


### 1. Load and Clean Historical Parking Sensor Records

In [ ]:
# Path to 2013 historical events (from pipeline schema)
csv_path = Path("../data/raw/parking_2013_extracted/On-street_Car_Parking_Sensor_Data_-_2013.csv")

if not csv_path.exists():
    print(f"Raw CSV not found at {csv_path}. Generating simulated parking event data for EDA demonstration.")
    # Simulate parking data matching the exact filter_supported_events.py schema
    dates = pd.date_range(start="2013-01-01", end="2013-02-28", freq="1min")
    arrivals = np.random.choice(dates, size=10000, replace=True)
    durations = np.random.exponential(scale=3600, size=10000).astype(int) # Seconds
    
    df_parking = pd.DataFrame({
        "DeviceId": np.random.randint(10000, 10100, size=10000),
        "StreetName": np.random.choice(["LA TROBE STREET", "BOURKE STREET", "COLLINS STREET", "SWANSTON STREET"], size=10000),
        "BetweenStreet1": "Cross St A",
        "BetweenStreet2": "Cross St B",
        "ArrivalTime": arrivals,
        "DurationSeconds": durations
    })
    df_parking["DepartureTime"] = df_parking["ArrivalTime"] + pd.to_timedelta(df_parking["DurationSeconds"], unit='s')
else:
    df_parking = pd.read_csv(csv_path, parse_dates=["ArrivalTime", "DepartureTime"])

# Extract hour and compute duration in minutes
df_parking["ArrivalHour"] = df_parking["ArrivalTime"].dt.hour
df_parking["DurationMins"] = df_parking["DurationSeconds"] / 60

print(f"Loaded {len(df_parking)} parking events.")
display(df_parking.head(2))


### 2. Compute Hourly Occupancy Utilization Patterns

In [ ]:
# Group by Street and Hour to find average duration and event counts
occupancy_summary = df_parking.groupby(["StreetName", "ArrivalHour"]).agg(
    total_events=("DeviceId", "count"),
    avg_duration_mins=("DurationMins", "mean")
).reset_index()

# Visualize Utilization Profiles
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.lineplot(
    data=occupancy_summary, x="ArrivalHour", y="total_events", hue="StreetName", 
    marker="o", palette="Dark2", ax=axes[0]
)
axes[0].set_title("Hourly Parking Event Frequency by Street", fontsize=14)
axes[0].set_xlabel("Hour of Day (Arrival)")
axes[0].set_ylabel("Total Parking Events")
axes[0].grid(True, linestyle="--", alpha=0.5)

sns.lineplot(
    data=occupancy_summary, x="ArrivalHour", y="avg_duration_mins", hue="StreetName", 
    marker="s", palette="Dark2", ax=axes[1]
)
axes[1].set_title("Average Stay Duration by Street", fontsize=14)
axes[1].set_xlabel("Hour of Day (Arrival)")
axes[1].set_ylabel("Average Duration (Minutes)")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


### 3. Prototype Aerial Imagery Proxy Sampling Methodology

In [ ]:
# As outlined in DATA.md, for non-sensor LGAs we prototype a sampling methodology 
# to count parked cars in buffers before and after interventions using aerial proxies.

years = ["Before (Baseline)", "After (Post-Intervention)"]
sites = ["Site A (Intervention Corridor)", "Site B (Control Corridor)"]

# Simulate object detection or manual sampling counts
np.random.seed(42)
df_proxy = pd.DataFrame([
    {
        "Site": site, 
        "Period": period, 
        "Counted_Cars": np.random.randint(55, 85) if (period=="Before (Baseline)" or site=="Site B (Control Corridor)") else np.random.randint(35, 50), 
        "Total_Bays": 100
    }
    for site in sites for period in years
])

df_proxy["Utilization_Rate"] = df_proxy["Counted_Cars"] / df_proxy["Total_Bays"]

plt.figure(figsize=(8, 5))
sns.barplot(
    data=df_proxy, x="Site", y="Utilization_Rate", hue="Period", 
    palette="viridis", alpha=0.8
)
plt.title("Proxy Prototype: Parking Utilization via Aerial Imagery Counting", fontsize=14)
plt.ylabel("Occupancy Rate")
plt.xlabel("Study Corridor")
plt.ylim(0, 1.0)
plt.grid(True, linestyle="--", alpha=0.5, axis="y")
plt.legend(loc="upper right")
plt.show()
